In [5]:
import pandas as pd
import yfinance as yf
from datetime import date
from yfetch import get_stock_history
from time import sleep

def temperature(series):
    series = series.dropna()
    if len(series) == 0:
        return None
    return (series <= series.iloc[-1]).mean()

def run(symbol, *extra_expiries):
  history = get_stock_history(symbol, interval='1d', cache_days=2)
  print(f'{symbol} ${history["Close"].iloc[-1]:.2f}')
  sleep(1)  # to avoid hitting yfinance rate limits
  ticker = yf.Ticker(symbol)
  all_expiries = sorted(set(ticker.options) | set(extra_expiries))
  rows = []
  for expiry in all_expiries:
    calendar_days = (date.fromisoformat(expiry) - date.today()).days
    weeks = round(calendar_days / 7)
    if weeks < 1:
        continue
    sma_days = weeks * 5
    sma = history['Close'].rolling(window=sma_days).mean()
    sma_dist = history.Close / sma - 1
    T = temperature(sma_dist)
    P10 = sma.iloc[-1] * (1 + sma_dist.quantile(0.10))
    mean_change = history['Close'].pct_change(periods=sma_days).mean()
    rows.append({
        'expiry': expiry,
        'days': calendar_days,
        'SMA(d)': sma_days,
        'SMA': sma.iloc[-1],
        'dist': sma_dist.iloc[-1],
        'T': T,
        'P10': P10,
        'change': mean_change,
    })

  df = pd.DataFrame(rows)
  if df.empty:
    print(f"{symbol} above all option SMAs!")
    return
  for col in ['dist', 'T']:
      df[col] = df[col].map('{:.2%}'.format)
  df['SMA'] = df['SMA'].map('${:.2f}'.format)
  df['change'] = df['change'].map('{:.2%}'.format)
  df['P10'] = df['P10'].map('${:.2f}'.format)
  print(df)

In [7]:
run('USD')

Fetched history for USD (1256 rows)
USD $52.85
       expiry  days  SMA(d)     SMA    dist       T     P10  change
0  2026-03-20    20      15  $57.14  -7.50%  16.67%  $51.05   4.04%
1  2026-04-17    48      35  $56.45  -6.38%  24.22%  $47.03   9.53%
2  2026-05-15    76      55  $55.11  -4.11%  28.37%  $43.58  15.18%
3  2026-08-21   174     125  $52.75   0.19%  32.24%  $36.93  38.38%
